# Neural Network Role Scoring Model


In [13]:
import pandas as pd
import numpy as np
import os

BASE_DIR = os.path.abspath("..")

# Load role-skill knowledge base
role_skill_map = pd.read_csv(
    os.path.join(BASE_DIR, "data", "processed", "role_skill_map.csv")
)

# Load skill demand scores
skill_demand = pd.read_csv(
    os.path.join(BASE_DIR, "data", "processed", "skill_demand_scores.csv")
)

role_skill_map.head()


,Title,Role_Skills
0,.NET Developer,"['.net core basics', 'git', 'design patterns',..."
1,AI Engineer - Experienced,"['kubernetes', 'spark', 'deep learning', 'r', ..."
2,AI Engineer - Fresher,"['c++', 'r', 'pytorch', 'keras', 'nltk', 'java..."
3,AI Prompt Engineer,"['intro nlp', 'openai gpt models exposure', 't..."
4,AR/VR Developer,"['unreal engine basics', 'teamwork', 'vuforia'..."


In [14]:
role_skill_map["Role_Skills"] = role_skill_map["Role_Skills"].apply(eval)


In [15]:
demand_dict = dict(
    zip(skill_demand["Skill"], skill_demand["Demand_Score"])
)


In [16]:
training_df = role_skill_map.copy()

training_df["Num_Missing_Skills"] = training_df["Role_Skills"].apply(len)

training_df["Avg_Demand_Score"] = training_df["Role_Skills"].apply(
    lambda skills: np.mean(
        [demand_dict.get(skill, 0) for skill in skills]
    )
)

training_df.head()


,Title,Role_Skills,Num_Missing_Skills,Avg_Demand_Score
0,.NET Developer,"[.net core basics, git, design patterns, html,...",44,0.103409
1,AI Engineer - Experienced,"[kubernetes, spark, deep learning, r, c++, pyt...",30,0.203788
2,AI Engineer - Fresher,"[c++, r, pytorch, keras, nltk, java, clusterin...",20,0.216591
3,AI Prompt Engineer,"[intro nlp, openai gpt models exposure, teamwo...",60,0.119773
4,AR/VR Developer,"[unreal engine basics, teamwork, vuforia, comm...",60,0.106818


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Convert role skills to text
role_skill_map["skill_text"] = role_skill_map["Role_Skills"].apply(
    lambda skills: " ".join(skills)
)

# Vectorize
vectorizer = TfidfVectorizer()
role_vectors = vectorizer.fit_transform(role_skill_map["skill_text"])

# Simulate a generic user (for training purposes)
user_skills = list(set(sum(role_skill_map["Role_Skills"].tolist(), [])))
user_skill_text = " ".join(user_skills)

user_vector = vectorizer.transform([user_skill_text])

# Compute match score
similarity_scores = cosine_similarity(user_vector, role_vectors)[0]

role_skill_map["Match_Percentage"] = (similarity_scores * 100).round(2)


In [18]:
def weighted_gap_score(role_skills):
    return sum(demand_dict.get(skill, 0) for skill in role_skills)

role_skill_map["Gap_Severity_Score"] = role_skill_map["Role_Skills"].apply(
    weighted_gap_score
)


In [19]:
training_df = role_skill_map.copy()

training_df["Num_Missing_Skills"] = training_df["Role_Skills"].apply(len)

training_df["Avg_Demand_Score"] = training_df["Role_Skills"].apply(
    lambda skills: np.mean(
        [demand_dict.get(skill, 0) for skill in skills]
    )
)

training_df.head()


,Title,Role_Skills,skill_text,Match_Percentage,Gap_Severity_Score,Num_Missing_Skills,Avg_Demand_Score
0,.NET Developer,"[.net core basics, git, design patterns, html,...",.net core basics git design patterns html kube...,16.48,4.550000,44,0.103409
1,AI Engineer - Experienced,"[kubernetes, spark, deep learning, r, c++, pyt...",kubernetes spark deep learning r c++ pytorch k...,12.72,6.113636,30,0.203788
2,AI Engineer - Fresher,"[c++, r, pytorch, keras, nltk, java, clusterin...",c++ r pytorch keras nltk java clustering pytho...,6.44,4.331818,20,0.216591
3,AI Prompt Engineer,"[intro nlp, openai gpt models exposure, teamwo...",intro nlp openai gpt models exposure teamwork ...,25.19,7.186364,60,0.119773
4,AR/VR Developer,"[unreal engine basics, teamwork, vuforia, comm...",unreal engine basics teamwork vuforia communic...,38.05,6.409091,60,0.106818


In [20]:
training_df["Target_Score"] = (
    training_df["Match_Percentage"]
    - training_df["Gap_Severity_Score"]
)

training_df[[
    "Title",
    "Match_Percentage",
    "Gap_Severity_Score",
    "Target_Score"
]].head()


,Title,Match_Percentage,Gap_Severity_Score,Target_Score
0,.NET Developer,16.48,4.550000,11.930000
1,AI Engineer - Experienced,12.72,6.113636,6.606364
2,AI Engineer - Fresher,6.44,4.331818,2.108182
3,AI Prompt Engineer,25.19,7.186364,18.003636
4,AR/VR Developer,38.05,6.409091,31.640909


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = [
    "Match_Percentage",
    "Gap_Severity_Score",
    "Num_Missing_Skills",
    "Avg_Demand_Score"
]

X = training_df[features]
y = training_df["Target_Score"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [22]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

mlp.fit(X_scaled, y)


c:\Users\devan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(32, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",500
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [23]:
training_df["NN_Score"] = mlp.predict(X_scaled)
training_df.sort_values(by="NN_Score", ascending=False).head()


,Title,Role_Skills,skill_text,Match_Percentage,Gap_Severity_Score,Num_Missing_Skills,Avg_Demand_Score,Target_Score,NN_Score
65,Digital Marketing Specialist,"[advanced digital marketing campaigns, teamwor...",advanced digital marketing campaigns teamwork ...,66.11,4.081818,138,0.029578,62.028182,61.862292
45,Copywriter,"[communication, content editing and proofreadi...",communication content editing and proofreading...,64.40,4.554545,83,0.054874,59.845455,59.310306
44,Content Writer,"[wordpress basics, fact-checking and research,...",wordpress basics fact-checking and research ai...,62.30,3.000000,101,0.029703,59.300000,59.247347
126,Marketing Specialist,"[multi-channel campaigns, lead generation and ...",multi-channel campaigns lead generation and mu...,60.13,2.727273,106,0.025729,57.402727,57.441411
150,SEO Specialist,"[advanced content optimization, ai and machine...",advanced content optimization ai and machine l...,49.70,1.140909,90,0.012677,48.559091,48.930259


In [24]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(mlp, "../models/nn_role_ranker.pkl")
joblib.dump(scaler, "../models/feature_scaler.pkl")


['../models/feature_scaler.pkl']